<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.2-channel-flow/Ex09.2_hydrogen_channel.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.2 — Hydrogen Along a Fuel Channel

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L9.2 · Gas Flow**

Humidified hydrogen flows along the fuel channel of a solid oxide fuel cell at
800 °C. Along the electrode wall hydrogen is taken up and steam returned, at a
rate set by the current. How does the gas change along the channel and across
it, how much of the fuel is used, and how fast must the gas be fed so that the
electrode at the outlet still sees enough hydrogen? It is the species
equation of L9.2 on a flow that is known in closed form, and beyond the first
millimetre its solution is **exact**: linear along the channel, a quartic
across it, with a Sherwood number that a handbook tabulates. A marching grid
and a physics-informed network are scored against that truth. The network
takes the inlet speed as an input, so one training answers for every speed,
and its gradient finds the speed a design asks for. The electrode itself -
what happens inside it, and the voltage it makes - is L10.2's subject; here
the current is a given number.

**The same channel serves a fuel cell and an electrolyser.** A solid oxide
cell runs both ways. As a fuel cell (SOFC) its electrode takes hydrogen from
this channel and returns steam, and power comes out; driven backwards by a
power supply as an electrolyser (SOEC), the electrode takes steam and returns
hydrogen. The figure after the setup cells shows the cell behind the channel
working both ways. For the channel the only difference is the sign of the
flux at the wall, so everything below holds for both: this notebook runs the
fuel cell, and mini project MP9.2A the electrolyser.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics: the gas, the flow, the species, the exact solution | — |
| **4** | finite differences marched along the channel, refined until an agreed 0.1 percentage point | — |
| **5** | one PINN for every inlet speed, and the comparison | the trial function, the residual |
| **6** | the inlet speed a design asks for, by a gradient through the network | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About six minutes on a CPU, most of it the network's training. Numbers are
printed with at most two decimals, fractions in per cent.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'channel.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-channel-flow/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "channel"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "channel.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import channel as ch
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex09.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

def online(fn, reps=5):                                    # the same, with the GPU made to finish before the clock stops
    def run():
        fn()
        if DEVICE.type == "cuda": torch.cuda.synchronize()
    return middle(run, reps)

**The cell behind the channel, both ways.** Left, the fuel cell: oxide ions
cross the electrolyte and burn the hydrogen that diffuses into the porous fuel
electrode, the steam returns to the channel, and the electrons drive a load.
Right, the same cell as an electrolyser: a power supply drives every arrow
the other way, steam is split and hydrogen returns. Section 2 says why each
is fed a little of the other gas.

In [ ]:
fig = ch.draw_cell(); plt.show()                              # the cell behind the wall, as a fuel cell and as an electrolyser

---

## 1 · The problem

A solid oxide cell: an electrode 100 × 100 mm at 800 °C and 1 atm, with a
fuel channel 1 mm high along it. The gas entering is 97 % hydrogen and 3 %
steam, at a mean speed $u$ between 0.3 and 1.5 m/s. The cell draws
0.5 A/cm², evenly along the electrode; for every two electrons one molecule
of hydrogen is taken from the gas at the wall and one molecule of steam is
returned. The opposite wall, the interconnect, is impermeable. What is the
steam fraction at every point of the channel, what share of the fuel is used,
and at what inlet speed does the hydrogen fraction at the electrode at the
outlet fall to 25 %?

## 2 · The data

| | |
|---|---|
| channel | 100 mm long, 1 mm high, per metre of width; 800 °C, 1 atm, so $c = p/RT$ = 11.36 mol/m³ |
| gas in | 97 % H₂, 3 % H₂O; density 0.028 kg/m³; viscosity 2.2e-05 Pa s — *typical value* |
| diffusivity | steam in hydrogen at 800 °C, $D$ = 8.0e-04 m²/s — *an estimate* |
| current | 0.5 A/cm², uniform: $N = i/2F$ = 0.0259 mol/(m² s) taken and returned at the electrode |
| inlet speed | 0.3 to 1.5 m/s; 1 m/s is the reference |

Two estimates and a typical value; every result below depends on them.

**What happens behind the wall.** The electrode does not soak up hydrogen. A
solid oxide cell is three layers (the figure at the top): the porous **fuel
electrode** on this channel, nickel and ceramic; a thin solid **electrolyte**
that conducts oxide ions O²⁻ and nothing else; and the **air electrode** on
the far side, with an air channel behind it. In the fuel cell (SOFC) oxygen
from the air takes two electrons at the air electrode, crosses the
electrolyte as an oxide ion, and meets the hydrogen that has diffused into
the pores of the fuel electrode:

$$\mathrm{H_2 + O^{2-} \;\to\; H_2O + 2e^-}$$

The two electrons go round the external circuit through the load - that is
the current $i$ - and the steam diffuses back into the channel. So for every
two electrons one molecule of hydrogen leaves the gas and one of steam enters
it: the wall flux $N = i/2F$ of the table, equal and opposite. This notebook
sees only that exchange; the voltage it makes, and what limits it inside the
electrode, come in L10.2.

**Why steam is fed with the hydrogen.** The reaction does not need steam - it
makes it. The 3 % at the inlet is humidification. The cell's voltage depends
on the ratio of hydrogen to steam at the electrode (L10.2's Nernst potential,
through $\ln(y_{\mathrm{H_2}}/y_{\mathrm{H_2O}})$); with no steam at all that
ratio is infinite at the inlet, and the open-circuit voltage is not defined
and not steady. So a little steam is always fed; a few per cent is usual for
laboratory cells - *a typical value, not from a cited source*.

**The same cell as an electrolyser (SOEC).** Driven by a power supply above
its open-circuit voltage, the cell runs backwards (the right of the figure at the top):
steam is split at the fuel electrode, $\mathrm{H_2O + 2e^- \to H_2 + O^{2-}}$,
the oxide ions cross the electrolyte the other way and leave the air
electrode as oxygen, and the hydrogen goes back into the channel. Steam is
now the reactant and hydrogen the product, so the feed is mostly steam, with
some hydrogen - about 10 % - to keep the nickel of the electrode from
oxidising where the gas enters (*typical, not cited*). For the channel the
only change is the sign of the wall flux: that is mini project MP9.2A, and
Ex_10.2 runs that electrolyser with its voltage.

## 3 · The physics

**Three facts from L9.2 make the gas simple.** It is an ideal gas at a Mach
number of 1e-03, so the flow is incompressible. The exchange at the wall is
one molecule for one, so the molar concentration $c$ stays what $p/RT$ says
and the velocity field is unchanged by the reaction - even though the mass
density rises as light hydrogen becomes heavy steam, which is why the species
is counted in **mole fractions**. And the Reynolds number is 1.3 with an
entrance length of 0.06 mm, so the velocity is **Poiseuille's parabola** from
the inlet, $u(z) = 6\,u\,(z/h)(1 - z/h)$, and the pressure drop is
$12\,\mu\,u\,L/h^2$, 26 Pa at 1 m/s.

**Which velocity is there.** The velocity has three components: $u$ along
the channel, $v$ across the 1 mm gap, $w$ across the 100 mm width. No gas
passes through either wall and, with the flow unchanging along the channel,
the volume condition $\nabla\cdot\mathbf{u} = 0$ leaves no flow across the
gap: $v = 0$. The channel is a hundred times wider than it is high, so away
from its two edges nothing varies across the width and nothing drives a flow
that way: $w = 0$. What is left is $u(z)$, the parabola, the same at every
$x$; the inertia term $(\mathbf{u}\cdot\nabla)\mathbf{u}$ of L9.1 is then
exactly zero, as in Ex_09.1's tube, and the flow is solved before the
notebook starts. Its unknown is the steam the flow carries.

In [ ]:
ch.draw_coordinates(); plt.show()                             # x along, z across the gap, the width into the page; u kept, v and w zero

**The species equation** carries the steam fraction $y$ along by the flow and
spreads it across by diffusion (L9.2's *Convection–Diffusion*):

$$u(z)\,y_x = D\,y_{zz}, \qquad
y = y_{\mathrm{in}}\ \text{at}\ x = 0, \qquad
-D c\,y_z = N\ \text{at}\ z = 0, \qquad
y_z = 0\ \text{at}\ z = h$$

Diffusion along the channel is dropped: the Péclet number along it is 125,
so that term is a ten-thousandth of the one across, and dropping it is what
the Graetz problem does. What is left is **parabolic in $x$**: it marches from
the inlet the way the heat equation of L8.2 marches in time, and it needs no
outlet condition. (Kept, the term would need a fully developed outflow,
$y_{xx} = 0$; a zero-slope outlet would be wrong, because the mean keeps
rising to the last millimetre.)

**The hand calculation** is the integral balance: the steam the wall returns
over a length $x$ is carried away by the flow, so the mixing-cup mean rises
linearly, $\bar y(x) = y_{\mathrm{in}} + N x/(u h c)$. Fuel utilisation is
that rise at the outlet over the hydrogen fed: 23.5 % at 1 m/s, 78 % at
0.3 m/s. What the hand calculation misses is the **profile across the
channel**: the steam is made at one wall, and the electrode sees the wall
value, not the mean.

**Scaled units.** $X = x/L$, $Z = z/h$, the inlet speed as $s = u / 1\ \mathrm{m/s}$,
and the rise of the steam fraction over $Y = NL/(u_{\mathrm{ref}} h c)$ = 0.228,
the mixing-cup rise over the whole channel at 1 m/s: $\theta = (y - y_{\mathrm{in}})/Y$.
Then

$$6Z(1-Z)\,G\,s\,\theta_X = \theta_{ZZ}, \qquad
\theta = 0\ \text{at}\ X = 0, \qquad \theta_Z = -G\ \text{at}\ Z = 0, \qquad \theta_Z = 0\ \text{at}\ Z = 1$$

with one constant, $G = u_{\mathrm{ref}} h^2/(DL)$ = 0.0125, the Péclet number
across the channel times the aspect ratio.

**The exact solution.** Beyond an entrance a fraction of a millimetre long,
the field is linear along the channel and a quartic across it:

$$\theta(X, Z) = \frac{X}{s} + G\,\varphi(Z), \qquad
\varphi(Z) = Z^3 - \frac{Z^4}{2} - Z + \frac{13}{35}$$

$X/s$ is the hand calculation. $\varphi$ comes from integrating the equation
twice in $Z$ with the two wall conditions, the constant fixed by the balance
(the flow-weighted mean of $\varphi$ is zero). The wall sits $13G/35$ above
the mean.

**The Sherwood number** is how engineers tabulate that gap. A wall exchanging
a species with a flowing gas is described by a *mass-transfer coefficient*
$h_m$, the flux per unit of the difference between the wall and the mean of
the gas, $N = h_m\,c\,(y_{\mathrm{wall}} - \bar y)$ - the species twin of the
heat transfer coefficient $h$ of L8.1, with the mean of the gas playing the
part of the ambient temperature. Made dimensionless with the channel's
hydraulic diameter $D_h = 2h$ and the diffusivity,

$$\mathrm{Sh} = \frac{h_m D_h}{D} = \frac{N\,D_h}{D\,c\,(y_{\mathrm{wall}} - \bar y)}$$

it is the species twin of the Nusselt number, and in a fully developed
laminar channel it is a **constant of the geometry alone**: one number gives
the wall concentration from the mean for any speed, any gas, any flux. Here
$y_{\mathrm{wall}} - \bar y = 13 G Y/35$ and $N = D c\,G Y/h$ in these units,
so $\mathrm{Sh} = 2 \times 35/13$ = **5.385**, the constant Shah & London
tabulate for parallel plates with one wall at constant flux and the other
impermeable - Incropera's table 8.1 gives the same number for heat. That
published constant is the check on the derivation; the field is then exact
to rounding, and it is the truth the two methods below are scored against,
for $x$ beyond 5 mm. Section 4 reports the Sherwood number each method
implies, because it is the hardest thing in the field to get right: a tenth
of a per cent of the steam fraction. The cell prints the data, draws the
channel, and plots the exact field.

In [ ]:
ch.describe_problem()
fig, ax = plt.subplots(figsize=(10.5, 3.6))
ch.draw_channel(ax)
plt.tight_layout(); plt.show()

Zp = np.linspace(0, 1, 201); Xp = np.linspace(0, 1, 401)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for s, c in ((0.3, "tab:red"), (1.0, "tab:blue"), (1.5, "tab:green")):
    ax[0].plot(Xp * 100, 100 * ch.kelvin_free(ch.exact(Xp, 0.0, s)), color=c, lw=1.8, label=f"u = {s:.1f} m/s, at the electrode")
    ax[0].plot(Xp * 100, 100 * ch.kelvin_free(Xp / s), color=c, lw=1, ls="--")
ax[0].set_xlabel("x  [mm]"); ax[0].set_ylabel("steam fraction  [%]"); ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title("along the channel: at the electrode (solid) and the mixing-cup mean (dashed)", fontsize=9)
ax[1].plot(100 * ch.Y_SCALE * ch.G * ch.phi(Zp), Zp, lw=2)
ax[1].axvline(0, color="k", lw=0.8, ls=":"); ax[1].set_xlabel("steam fraction above the mixing-cup mean  [% points]"); ax[1].set_ylabel("z / h")
ax[1].set_title("across the channel: the exact profile φ, the same at every x and speed", fontsize=9)
plt.tight_layout(); plt.show()
print(f"the electrode sees {100 * ch.Y_SCALE * ch.G * ch.phi(0.0):.3f} percentage points more steam than the mean, at every x and speed; Sherwood number {ch.SHERWOOD:.3f}")
for s in (1.0, 0.3):
    y_w, h2 = ch.exact_wall_outlet(s)
    print(f"at {s:.1f} m/s: utilisation {100 * ch.utilisation(s):.1f} %, hydrogen at the electrode at the outlet {100 * h2:.2f} %, Δp = {ch.pressure_drop(s):.2f} Pa")

**What you should see.** The steam fraction rising in a straight line along
the channel, steeper the slower the gas: from 3 % to 26 % at 1 m/s and to
79 % at 0.3 m/s. The electrode sees 0.11 percentage points more steam than the
mean, the same at every $x$ and every speed - the profile across the channel
is a tenth of a per cent of the field. At 800 °C hydrogen diffuses so fast
that the channel is mixed across its height within a millimetre: the fuel
channel's problem is the depletion along it, not across it. What limits the
cell happens inside the electrode, which is L10.2's subject.

---

## 4 · Finite differences, marched from the inlet

**$x$ is stepped like time.** The equation is parabolic in $x$, so the grid
does not solve one large system: it marches, one column of the channel at a
time, from the inlet. Each step is **Crank–Nicolson** in $x$ (half the
diffusion at the old column, half at the new, second order) with central
differences across the channel, and the two walls by ghost nodes - the
electrode's flux through its ghost, the interconnect mirrored. At the walls
the velocity is zero, so the marching term vanishes there and the step
reduces to the wall condition itself (`ch.fdm`).

**The agreed accuracy is 0.1 percentage point** of the steam fraction, a
tenth of the rise an engineer cares about. The cell refines the grid at three
speeds, scored against the exact field for $x$ beyond 5 mm, and times one
march.

In [ ]:
TARGET = 1e-3                                                 # in the steam fraction: 0.1 percentage point
SPEEDS = (0.3, 1.0, 1.5)
rows_fdm = {}
for nx, nz in ((101, 11), (201, 21), (401, 41), (801, 81)):
    errs = []
    for s in SPEEDS:
        X, Z, th = ch.fdm(nx, nz, s)
        XX, ZZ = np.meshgrid(X, Z)
        beyond = XX >= 0.05                                   # the entrance left out: the exact solution holds from 5 mm on
        errs.append(ch.Y_SCALE * np.abs(th - ch.exact(XX, ZZ, s))[beyond].max())
    sec = online(lambda: ch.fdm(nx, nz, 1.0), reps=3)
    rows_fdm[(nx, nz)] = (max(errs), sec)
    print(f"  {nx:3d} x {nz:2d} nodes: worst error in the steam fraction at u = 0.3, 1.0, 1.5 m/s:",
          "  ".join(f"{e:.2e}" for e in errs), f"  one march {sec * 1e3:.1f} ms" + ("   <- meets 0.1 % point" if max(errs) <= TARGET else ""))
NX, NZ = 401, 41                                              # the coarsest grid that meets it at every speed

X, Z, th = ch.fdm(801, 81, 1.5); XX, ZZ = np.meshgrid(X, Z)
dev = ch.Y_SCALE * np.abs(th - ch.exact(XX, ZZ, 1.5)).max(axis=0)
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.semilogy(X * 100, dev, lw=1.5); ax.axvline(5, color="k", lw=0.8, ls=":"); ax.axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 0.1 % point")
ax.set_xlabel("x  [mm]"); ax.set_ylabel("|grid − exact|, steam fraction"); ax.set_title("801 x 81 at 1.5 m/s: the inlet corner's error, decaying along the channel", fontsize=9)
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

**What you should see.** The error falling by about four with each halving
of the spacing on the coarse grids - 8e-03, 2e-03, 6e-04 in the fraction at
0.3 m/s - then more slowly, as the plot shows why: at the inlet corner, where
the gas arriving at 3 % steam meets a wall already returning steam, the
gradient is infinite and the march starts with an error it sheds only slowly
along the channel; the sawtooth is Crank–Nicolson's own odd–even oscillation,
set off by that corner and damped only slowly. **401 × 41 nodes meet 0.1
percentage point at every speed, in a few milliseconds per speed.** Every
speed is a new march.

---

## 5 · One network for every inlet speed, and the comparison

**The inlet speed is an input.** The network takes $(X, Z, s)$ and is trained
once over $s$ from 0.3 to 1.5 (L9.2's *One Network for Every Flow Rate*).
After that any speed is read off, not re-solved - which is the one thing a
march cannot do.

**Build in what you know.** The hand calculation $X/s$ solves the equation
and the inlet; only the profile across the channel is missing, and it is a
hundredth of the field. So the trial function is

$$\hat\theta(X, Z, s) = \frac{X}{s} + G\,\mathcal{N}(X, Z, s)$$

and the network $\mathcal{N}$, four layers of 32 tanh neurons, learns the
profile in units of $G$, where it is of order one. The inlet and the two walls
are loss terms, each divided by $G$ so that they are of order one too; the
residual is divided by $G$ for the same reason. There is no outlet term,
because the equation has no outlet condition.

**TODO 1** is the trial function; **TODO 2** the residual of the scaled
equation, $(\hat\theta_{ZZ} - 6Z(1-Z)\,G\,s\,\hat\theta_X)/G$.

In [ ]:
def features(p):
    X, Z, s = p[:, 0:1], p[:, 1:2], p[:, 2:3]
    return torch.cat([X, Z, (s - 0.9) / 0.6], 1)              # the speed scaled to [-1, 1], like X and Z are of order one

class Channel(torch.nn.Module):                               # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=32, n_layers=4)       # dense tanh network: X, Z, s in
    def forward(self, p):
        X, s = p[:, 0:1], p[:, 2:3]
        # TODO 1: the trial function - the hand calculation X / s, plus ch.G times the network on features(p).
        #
        #   return X / s + ch.G * self.net(features(p))
        raise NotImplementedError("Write the trial function")

In [ ]:
# TODO 2: the residual of the scaled equation, (theta_ZZ - 6 Z (1 - Z) G s theta_X) / G.
#
#   def residual(model, p):
#       th = model(p)
#       g = grad(th, p)                                           # columns X, Z, s
#       th_zz = grad(g[:, 1:2], p)[:, 1:2]
#       return (th_zz - ch.poiseuille(p[:, 1:2]) * ch.G * p[:, 2:3] * g[:, 0:1]) / ch.G
raise NotImplementedError("Write residual(model, p)")

In [ ]:
W_F = 10.0                                                    # the weight of the inlet and the two walls
pts = ch.sample_channel(4000)                                 # (X, Z, s): a fifth of them in the first 5 mm, where the profile develops
inlet, electrode, interconnect, _ = ch.sample_faces(300)      # the three faces that carry a condition, at random speeds
P_in = to_tensor(pts, requires_grad=True)                     # collocation points, differentiable
P_inlet, P_el, P_ic = (to_tensor(q, requires_grad=True) for q in (inlet, electrode, interconnect))   # the faces, differentiable

def face_loss(model):
    at_inlet = model(P_inlet) / ch.G                                     # theta = 0 at the inlet, in units of G
    at_electrode = grad(model(P_el), P_el)[:, 1:2] / ch.G + 1.0          # theta_Z = -G at the electrode; grad = autograd
    at_interconnect = grad(model(P_ic), P_ic)[:, 1:2] / ch.G             # theta_Z = 0 at the interconnect
    return at_inlet.pow(2).mean() + at_electrode.pow(2).mean() + at_interconnect.pow(2).mean()

model = Channel().to(DEVICE)                                  # four layers of 32
describe(model.net, n_collocation=len(P_in))                  # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, P_in).pow(2).mean() + W_F * face_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
history = train_two_stage(model, loss_fn, adam_steps=1000, lbfgs_steps=300, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s (offline, once for every speed); residual {residual(model, P_in).pow(2).mean().item():.2e}, faces {face_loss(model).item():.2e}")

In [ ]:
X, Z = np.linspace(0, 1, NX), np.linspace(0, 1, NZ); XX, ZZ = np.meshgrid(X, Z)
beyond = XX >= 0.05
def pinn_field(s):
    with torch.no_grad():  # no gradient tracking: evaluation only
        return to_numpy(model(to_tensor(np.c_[XX.ravel(), ZZ.ravel(), np.full(XX.size, s)]))).reshape(XX.shape)   # the network on the grid at one speed
results = []
for s in SPEEDS:
    T_true = ch.exact(XX, ZZ, s)
    T_f = ch.fdm(NX, NZ, s)[2]; T_p = pinn_field(s)
    e_f = ch.Y_SCALE * np.abs(T_f - T_true)[beyond].max(); e_p = ch.Y_SCALE * np.abs(T_p - T_true)[beyond].max()
    sh_f = 2 / ((T_f[0, -1] - 1 / s) / ch.G); sh_p = 2 / ((T_p[0, -1] - 1 / s) / ch.G)   # the Sherwood number each implies at the outlet
    results.append((s, 1 - ch.kelvin_free(T_true[0, -1]), 1 - ch.kelvin_free(T_f[0, -1]), 1 - ch.kelvin_free(T_p[0, -1]), e_f, e_p, sh_f, sh_p))
t_fdm = online(lambda: ch.fdm(NX, NZ, 1.0), reps=3)           # one march, one speed, on this runtime
t_pin = online(lambda: pinn_field(1.0))                       # the trained network on the same nodes, one speed, on this runtime

# the time column is what runs ONLINE: one march at one speed against one evaluation of the trained network at one speed,
# on the same runtime. The training is offline, paid once for every speed, and reported on its own line.
print(f"{'u':>5s}{'hydrogen at the electrode at the outlet':>42s}{'worst error, steam fraction':>30s}{'Sherwood':>18s}")
print(f"{'m/s':>5s}{'exact':>14s}{'grid':>14s}{'network':>14s}{'grid':>15s}{'network':>15s}{'grid':>9s}{'network':>9s}")
for s, h_e, h_f, h_p, e_f, e_p, sh_f, sh_p in results:
    print(f"{s:5.1f}{100 * h_e:13.2f} %{100 * h_f:13.2f} %{100 * h_p:13.2f} %{e_f:15.2e}{e_p:15.2e}{sh_f:9.2f}{sh_p:9.2f}")
print(f"exact Sherwood number {ch.SHERWOOD:.3f}")
print(f"time online, one speed: grid {NX} x {NZ} ({NX * NZ} unknowns) {t_fdm * 1e3:.1f} ms; network ({parameter_count(model)} weights) {t_pin * 1e3:.1f} ms; "
      f"trained offline in {train_time:.0f} s" + (f", repaid after {train_time / (t_fdm - t_pin):.0f} speeds" if t_fdm > t_pin else ", never repaid on time alone"))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for s, c in ((0.3, "tab:red"), (1.5, "tab:green")):
    T_true = ch.exact(XX, ZZ, s); T_f = ch.fdm(NX, NZ, s)[2]; T_p = pinn_field(s)
    ax[0].semilogy(X * 100, ch.Y_SCALE * np.abs(T_f - T_true)[0], color=c, lw=1.5, label=f"grid, u = {s:.1f} m/s")
    ax[0].semilogy(X * 100, ch.Y_SCALE * np.abs(T_p - T_true)[0], color=c, lw=1.5, ls="--", label=f"network, u = {s:.1f} m/s")
ax[0].axhline(TARGET, color="k", lw=0.8, ls=":"); ax[0].set_xlabel("x  [mm]"); ax[0].set_ylabel("error at the electrode, steam fraction"); ax[0].legend(frameon=False, fontsize=7, ncol=2)
T_p = pinn_field(1.0)
ax[1].plot(100 * ch.Y_SCALE * ch.G * ch.phi(Z), Z, "k", lw=2, label="exact")
ax[1].plot(100 * ch.Y_SCALE * (ch.fdm(NX, NZ, 1.0)[2][:, -1] - 1.0), Z, "s", ms=4, label=f"grid {NX} x {NZ}")
ax[1].plot(100 * ch.Y_SCALE * (T_p[:, -1] - 1.0), Z, "o", ms=3, label="network")
ax[1].set_xlabel("steam above the mean at the outlet, 1 m/s  [% points]"); ax[1].set_ylabel("z / h"); ax[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(cc.output_path("Ex09.2_report.png"), dpi=150); plt.show()

**Time online at the same accuracy.** The table compares the network with the
401 × 41 grid, which is less exact than the network. A fair race asks how
fine the grid must be to be **as exact as the network**, and what one answer
then costs each of them online - the training is offline and is left out.
The cell refines the grid six times at 1 m/s, times one march of each and
scores it against the exact field, then times the trained network's
**inference only**: once for the whole field on the 401 × 41 nodes, and once
for the single number section 6 needs, the hydrogen at the electrode at the
outlet. A grid has to march the whole channel to reach the outlet; the
network evaluates the point it is asked about.

In [ ]:
# online time at comparable accuracy: the grid refined step by step against the trained network's inference, one speed (1 m/s), this runtime
GRIDS = [(51, 6), (101, 11), (201, 21), (401, 41), (801, 81), (1601, 161)]
bars = []
for nx, nz in GRIDS:
    Xg, Zg, Tg = ch.fdm(nx, nz, 1.0); XXg, ZZg = np.meshgrid(Xg, Zg)
    err = ch.Y_SCALE * np.abs(Tg - ch.exact(XXg, ZZg, 1.0))[XXg >= 0.05].max()   # worst error in the steam fraction beyond 5 mm
    bars.append((f"grid {nx} x {nz}", online(lambda: ch.fdm(nx, nz, 1.0), reps=3), err, "grid"))
e_net = ch.Y_SCALE * np.abs(pinn_field(1.0) - ch.exact(XX, ZZ, 1.0))[beyond].max()   # the network's worst error, same scoring
outlet = to_tensor(np.array([[1.0, 0.0, 1.0]]))               # the one point a design asks about: X = 1, Z = 0 (the electrode), s = 1
def pinn_point():
    with torch.no_grad():                                     # no gradient tracking: evaluation only
        return model(outlet)                                  # one forward pass through the trained network
bars.append((f"network, field {NX} x {NZ}", online(lambda: pinn_field(1.0)), e_net, "net"))
bars.append(("network, one point", online(pinn_point), e_net, "net"))

match = [b for b in bars if b[3] == "grid" and b[2] <= e_net]     # the grids at least as exact as the network
ref = match[0] if match else [b for b in bars if b[3] == "grid"][-1]
fig, ax = plt.subplots(figsize=(9, 3.8))
for i, (name, tt, err, kind) in enumerate(bars):
    col = "tab:green" if kind == "net" else ("tab:blue" if err <= e_net else "0.7")
    ax.barh(i, tt * 1e3, color=col)
    ax.text(tt * 1e3 * 1.15, i, f"{tt * 1e3:.2f} ms,  error {err:.1e}", va="center", fontsize=8.5)
ax.set_yticks(range(len(bars))); ax.set_yticklabels([b[0] for b in bars]); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xlabel("time online for one answer at 1 m/s  [ms]  (log scale; the training is not included)")
ax.set_xlim(right=ax.get_xlim()[1] * 8)
ax.set_title("blue: grids at least as exact as the network;  grey: less exact;  green: the trained network", fontsize=9)
plt.tight_layout(); plt.show()
print(f"network's error {e_net:.1e}; the coarsest grid as exact is {ref[0]} ({ref[2]:.1e}), {ref[1] * 1e3:.2f} ms" if match else
      f"network's error {e_net:.1e}; no grid here is as exact - the finest, {ref[0]}, has {ref[2]:.1e} in {ref[1] * 1e3:.2f} ms")
print(f"at that accuracy the network is {ref[1] / bars[-2][1]:.1f} times faster for the whole field "
      f"and {ref[1] / bars[-1][1]:.0f} times faster for one number; trained once, offline, in {train_time:.0f} s")

**What you should see.** Both methods within 0.1 percentage point of the
exact steam fraction at every speed beyond the first 5 mm, and both agreeing
with the exact hydrogen fraction at the electrode at the outlet to a few
hundredths of a point: 74.08 % at 1 m/s, 20.84 % at 0.3 m/s. The network is
a little more exact than the 401 × 41 grid at every speed, and it is one
network for every speed where the grid marched three times. The fine
structure shows what each resolves: the Sherwood number measures the profile
across the channel, a tenth of a per cent of the field, and neither gets it
well - the grid within 1 to 25 % of 5.385, worst at the slowest speed where
the inlet corner's error has decayed least, the network 10 to 20 % high at
every speed, because it resolves the field to a thousandth and the profile on
top of it only to a tenth. (The network's numbers vary a little from run to
run; the grid's do not.)

**By accuracy** the order is exact, network, grid, with the two close. **By
time online at the same accuracy** (the bar chart) the grid has to be refined
to 801 × 81 nodes before it is as exact as the network, and one march then
takes about 8 ms on a laptop CPU; the trained network returns the whole field
in about 1.5 ms, some five times faster, and the one number a design asks
for, the hydrogen at the electrode at the outlet, in about 0.1 ms, some
seventy times faster - because the grid has to march the whole channel to
reach the outlet and the network evaluates only the point it is asked about.
The gain is smaller than in a 2-D or 3-D field, because this grid is cheap: it
marches one short column at a time, so halving its spacing only doubles its
time. The training, about two minutes on the CPU, is the price, and at a few
milliseconds saved per answer it is repaid only after some twenty thousand
answers. So the network pays where the same question is asked many times -
and above all through **the speed as an input**: the next section asks it a
question the grid can only answer by being run again and again. (Timings vary
from run to run and from runtime to runtime; the ratios are what to read.)

---

## 6 · The inlet speed a design asks for

A cell is not run at a given speed; it is run so that the electrode never
starves. Suppose the design rule is **25 % hydrogen at the electrode at the
outlet**: the slowest feed, and so the highest utilisation, that keeps that
margin. The network gives the hydrogen fraction there as a function of the
speed, $\hat\theta(1, 0, s)$, and `torch.autograd` gives its derivative with
respect to $s$ - so Newton's method, started from the hand calculation's own
speed $1/\theta$, finds the speed in a few steps, with no further solve. The
exact answer is in closed form here, which is the check;
for a channel with no exact solution the same lines would be the only way
short of a sweep.

In [ ]:
H2_TARGET = 0.25                                              # hydrogen fraction at the electrode at the outlet
theta_target = (1 - H2_TARGET - ch.Y_IN) / ch.Y_SCALE         # the same, as a scaled rise
s_var = torch.tensor([[1.0 / theta_target]], dtype=torch.get_default_dtype(), device=DEVICE, requires_grad=True)   # start from the hand calculation's speed
for it in range(5):
    p = torch.cat([torch.ones_like(s_var), torch.zeros_like(s_var), s_var], 1)   # the point (X = 1, Z = 0, s)
    f = model(p) - theta_target                               # the misfit at the electrode's outlet
    df = torch.autograd.grad(f, s_var)[0]                     # its slope with respect to the speed, by autograd
    with torch.no_grad():  # no gradient tracking: the Newton update itself
        s_var -= f / df
s_net = float(s_var.detach())                               # the speed found, as a number
s_exact = 1.0 / (theta_target - ch.G * ch.phi(0.0))
print(f"the speed for {100 * H2_TARGET:.0f} % hydrogen at the electrode at the outlet: network {s_net:.4f} m/s, exact {s_exact:.4f} m/s "
      f"({100 * (s_net / s_exact - 1):+.2f} %); utilisation {100 * ch.utilisation(s_exact):.1f} %, "
      f"Δp = {ch.pressure_drop(s_exact):.2f} Pa, pumping power {ch.pressure_drop(s_exact) * s_exact * ch.U_REF * ch.H_CHANNEL * 1e3:.2f} mW per metre of width")
print(f"the hand calculation alone, 75 % utilisation, would say {ch.speed_for_utilisation(0.75):.4f} m/s: the profile across the channel costs {100 * (s_exact / ch.speed_for_utilisation(0.75) - 1):.1f} % more gas")

**What you should see.** The speed found by the network within a fraction of
a per cent of the exact 0.3174 m/s, in five Newton steps and no solve; a
utilisation of 74 %; a pressure drop of 8 Pa and a pumping power of a few
milliwatts per metre of width, nothing beside the cell's output. The hand
calculation would have said 0.3136 m/s: the profile across the channel costs
about one per cent more gas than the mean suggests - small, and exactly what
the two-dimensional solution adds.

---

## 7 · What the notebook says

* **The fuel channel is a one-dimensional problem with a small
  two-dimensional correction.** At 800 °C the gas mixes across the channel
  within a millimetre; the steam fraction rises along the channel and hardly
  varies across it, by 0.1 percentage point at the electrode. The cell's fuel
  limitation lives in the electrode and in the depletion along the channel,
  not across it.
* **The exact solution is a textbook result, and it kept the comparison
  honest.** Its Sherwood number is the published 5.385; both methods were
  scored against the field, and both meet 0.1 percentage point. The grid pays
  for the inlet corner, the network for the fine structure it was asked to
  learn on top of the hand calculation; neither gets the Sherwood number to
  better than ten per cent, because it is a tenth of a per cent of the field.
* **Dropping axial diffusion made the equation parabolic**, so the grid
  marches and needs no outlet condition - and a zero-slope outlet, had the term
  been kept, would have been wrong.
* **The network's gain is the speed as an input, not the time per speed.**
  Online both answer in milliseconds, and the march is the faster; the
  training is repaid only in the design question, which the network answers
  by a gradient and the grid by a sweep.
* **What the air channel and the electrolyser change is the sign and the size
  of the same terms:** the mini projects.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex09.2_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L9.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Which three facts of L9.2 reduced the gas to an incompressible, one-species problem on a known velocity, and which number justified each? (-> L9.2 Q1)": """
""",
    "Why was diffusion along the channel dropped, what did that do to the type of the equation, and what outlet condition did the march therefore need? (-> L9.2 Q2)": """
""",
    "What did the trial function build in and what was left for the network, and why did both methods find the field to a thousandth but the Sherwood number only to ten per cent or worse? (-> L9.2 Q3)": """
""",
    "In section 6, what question did the network answer that the grid could not answer without a sweep, and how did autograd enter? (-> L9.2 Q4)": """
""",
    "To conclude: for this channel, which method would you use for one speed, and what would have to be true of a channel for the network's training to be worth it? (-> L9.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| u [m/s] | hydrogen at the electrode at the outlet: exact / grid / network | worst error, steam fraction: grid / network | Sherwood: grid / network |", "|---|---|---|---|"]
    for s, h_e, h_f, h_p, e_f, e_p, sh_f, sh_p in results:
        rows.append(f"| {s:.1f} | {100 * h_e:.2f} % / {100 * h_f:.2f} % / {100 * h_p:.2f} % | {e_f:.2e} / {e_p:.2e} | {sh_f:.2f} / {sh_p:.2f} |")
    rows += ["", f"Exact Sherwood number {ch.SHERWOOD:.3f}. Time online, one speed: grid {t_fdm * 1e3:.1f} ms, network {t_pin * 1e3:.1f} ms; training offline {train_time:.0f} s.",
             f"Design: the speed for {100 * H2_TARGET:.0f} % hydrogen at the electrode at the outlet, network {s_net:.4f} m/s, exact {s_exact:.4f} m/s."]
    out = ["# Ex_09.2 — Hydrogen Along a Fuel Channel", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex09.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex09.2_report.md into Ex09.2_report.pdf, with any figure
# saved as Ex09.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex09.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex09.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex09.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex09.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP9.2A · The SOEC steam channel

**The problem.** Run the same cell as an electrolyser: 90 % steam is fed (a typical feed),
steam is taken at the electrode and hydrogen returned, the same 0.5 A/cm². How
fast must the steam be fed for 70 % of it to be converted, and how pure is the
hydrogen at the outlet, at the electrode and in the mean?

| | |
|---|---|
| channel, gas, current | as in this notebook, the wall flux reversed |
| feed | 90 % steam, 10 % hydrogen |
| asked | the feed speed for 70 % conversion; the hydrogen fraction at the outlet, mean and at the electrode |

**Why a plain PINN is not enough.** Nothing new in the equation - the flux changes sign - so a plain PINN is enough,
and that is the point: show that the network of this notebook, retrained with
the sign and the feed changed, answers the electrolyser's question, and find the
conversion by a gradient through it as section 6 did the speed.

**What you build.** The network of this notebook with the reversed flux, trained over the same range
of speeds; the speed for 70 % conversion by Newton's method through the network;
the hydrogen fraction along the electrode.

**The ground truth you get.** `MP9.2A_truth.npz`, from `tools/miniprojects/ex092_truth.py`: the exact
solution of this notebook with the sign reversed, at every speed.

**Worked example.** 

| feed speed | steam converted | hydrogen at the outlet, mean | at the electrode |
|---|---|---|---|
| 1.00 m/s | 25.4 % | 32.8 % | 32.92 % |
| 0.50 m/s | 50.7 % | 55.6 % | 55.74 % |
| 0.362 m/s | 70.0 % | 73.0 % | 73.13 % |

**Requirements.**

* The speed for 70 % conversion within **1 %** of the exact value.
* The hydrogen fraction at the electrode within **0.1 percentage point** along the channel.
* One training for the whole range of speeds.

**What you hand in.** The hydrogen fraction along the electrode against the exact solution at three speeds, the speed found by the gradient, and a paragraph on what changed from the fuel cell and what did not.

### MP9.2B · The air channel

**The problem.** The other side of the fuel cell: air along the cathode channel, oxygen taken at
the electrode and nothing returned. Air is fed far in excess of what the current
needs, because it also cools the stack - a stoichiometry of five. The
diffusivity of oxygen in nitrogen is four times smaller than steam's in
hydrogen, so the profile across the channel is eleven times larger. How much
oxygen does the electrode see at the outlet, and does the one-sided exchange -
the small velocity it gives the gas at the wall - matter?

| | |
|---|---|
| channel, current | as in this notebook |
| gas | air, 21 % oxygen; $D$ = 1.9e-04 m²/s (O₂ in N₂ at 800 °C, estimate); viscosity 4.4e-05 Pa s (estimate) |
| exchange | oxygen taken at $i/4F$, nothing returned: a wall velocity $N/c$ into the electrode |
| feed | stoichiometry 5 |

**Why a plain PINN is not enough.** The one-sided exchange gives the gas a normal velocity at the wall, so the
velocity field is no longer Poiseuille's and the species equation gains a term.
Small here - the student first shows how small - but the network has to carry it
to be right, and a plain PINN on Poiseuille's profile would not.

**What you build.** First the exact solution of this notebook with the air's numbers, to see the size
of the profile; then the wall velocity estimated against the diffusion velocity;
then a PINN with the Stefan term added to the species equation and the wall
condition, scored against the exact solution without it, to show whether the term
changes the oxygen at the electrode by more than the agreed 0.1 percentage point.

**The ground truth you get.** `MP9.2B_truth.npz`, from `tools/miniprojects/ex092_truth.py`: the exact
solution with the air's numbers, Stefan flow left out, and the estimated wall velocity.

**Worked example.** At stoichiometry 5 the air runs at 2.72 m/s (Re 20, Δp 143 Pa); $G$ = 0.143, eleven times the fuel side's. Oxygen at the outlet: 16.80 % in the mean, 16.58 % at the electrode. The wall velocity $N/c$ is 1.14 mm/s against a diffusion velocity $D/h$ of 190 mm/s: 0.6 %.

**Requirements.**

* The oxygen fraction at the electrode within **0.1 percentage point** of the exact solution along the channel.
* A stated answer, with the number, to whether the Stefan term matters at this current.
* The air speed for 15 % oxygen at the electrode at the outlet, by a gradient through the network.

**What you hand in.** The oxygen fraction across the channel at the outlet, exact against network, with and without the Stefan term; the speed found; a paragraph on why the air side is fed five times what it needs.